# Leela Interp Demo with NNsight

## Setup

In [ ]:
import sys
import os

# Import modules from parent and sibling directories
parent_dir = os.path.abspath('..') 
leela_pytorch_impl_dir = os.path.join(parent_dir, 'leela_pytorch_impl')

if parent_dir not in sys.path:
    sys.path.append(parent_dir)
if leela_pytorch_impl_dir not in sys.path:
    sys.path.append(leela_pytorch_impl_dir)

from model import Lc0Model
from leela_board import LeelaBoard
import torch
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score

In [ ]:
ONNX_MODEL_NAME = 'lc0-original.onnx'
lc0 = Lc0Model(onnx_model_path=os.path.join(leela_pytorch_impl_dir, ONNX_MODEL_NAME))

Using device: cpu


## Dataset

We will use a sample chess dataset to run our first probe on the model

In [ ]:
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()  # Load environment variables from .env file

login(token=os.getenv("HUGGINGFACE_TOKEN"))

### Hugging Face Authentication

We load local `.env` variables and initialize our token to fetch the chess evaluation sets from Hugging Face properly.

In [ ]:
from datasets import load_dataset

print("1. Load actual dataset (streaming)")
try:
    # Trying lowercase to see if it fixes the typo
    data = load_dataset("lichess/chess-position-evaluations", split="train", streaming=True)
except Exception as e:
    print(f"Failed to load: {e}")

### Streaming the Dataset & Filtering

We stream the dataset from Hugging Face instead of downloading it fully into memory to prevent crashing. We also selectively filter the incoming stream to extract exactly 100 "in-check" positions and 100 "not-in-check" positions, guaranteeing a perfectly balanced dataset for our Logistic Regression probe.

In [ ]:

print("2. Extract Boards and Labels (Balanced)")
boards = []
labels_is_check = []

target_per_class = 100
num_check = 0
num_not_check = 0

for i, item in enumerate(data):
    if num_check >= target_per_class and num_not_check >= target_per_class:
        break
        
    fen = item.get('FEN', item.get('fen'))
    board = LeelaBoard.from_fen(fen)
    is_check = board.pc_board.is_check()
    
    if is_check and num_check < target_per_class:
        boards.append(board)
        labels_is_check.append(True)
        num_check += 1
    elif not is_check and num_not_check < target_per_class:
        boards.append(board)
        labels_is_check.append(False)
        num_not_check += 1

print(f"Found {num_check} in-check and {num_not_check} not-in-check positions.")

# Convert labels to numpy array
labels = np.array(labels_is_check, dtype=int)


In [ ]:

print("3. Force model and inputs to CPU")
lc0.cpu() # ensure everything is on CPU to avoid device mismatch
lc0._device = 'cpu'
inputs = lc0.make_inputs(boards).cpu()


## Extracting Activations

Here we construct inputs for the model and run the PyTorch forward pass. We use NNsight to intercept the outputs of the Multi-Layer Perceptron (MLP) at our designated probe layer (`LAYER_TO_PROBE = 7`). 

In [33]:
from nnsight import NNsight

print("Run forward pass and extract activations with NNsight...")
LAYER_TO_PROBE = 1

# Wrap our model in NNsight
nnsight_model = NNsight(lc0)

with nnsight_model.trace(inputs) as tracer:
    # Grab the output of the target layer and save it
    hidden_states = nnsight_model._lc0_model.post_mlp[LAYER_TO_PROBE].output.save()

print("Extracting features")
# `.value` was either removed or `hidden_states` is already evaluated to the tensor
X_activations = hidden_states if isinstance(hidden_states, torch.Tensor) else hidden_states.value
print(f"Activations shape: {X_activations.shape}")

if X_activations.ndim > 2:
    X_flattened = X_activations.reshape(X_activations.size(0), -1).cpu().numpy()
else:
    X_flattened = X_activations.cpu().numpy()

Run forward pass and extract activations with NNsight...


c:\Users\root\projects\chess-interp\.venv\Lib\site-packages\onnx2torch\node_converters\slice.py:63: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\python_variable_indexing.cpp:353.)
  x = x[pos_axes_slices]


Extracting features
Activations shape: torch.Size([200, 64, 768])


## Training the Linear Probe

After capturing the activations from our hidden layer, we split the data and train a Logistic Regression classifier (`Linear Probe`) with 5-fold cross-validation on the intermediate layer's geometry to see if it implicitly "understands" the concept of 'check'.

In [34]:
print("4. Train the linear probe (with Cross-Validation)")
probe = LogisticRegression(max_iter=1000)

# Split data into train and test sets
indices = np.arange(len(labels))
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_flattened, labels, indices, test_size=0.2, random_state=42, stratify=labels
)

if len(np.unique(y_train)) > 1:
    # Perform 5-fold cross validation on the training set
    cv_scores = cross_val_score(probe, X_train, y_train, cv=5)
    print(f"5-Fold CV Accuracy on train set: {cv_scores.mean():.2f} ± {cv_scores.std():.2f}")
    
    # Train the final probe on the full training set
    probe.fit(X_train, y_train)
    
    # Evaluate on the unseen test set
    test_acc = probe.score(X_test, y_test)
    print(f"Probe accuracy on hidden test set: {test_acc:.2f}")
else:
    print(f"Skipped LogisticRegression as there is only 1 class in the training sample.")

4. Train the linear probe (with Cross-Validation)
5-Fold CV Accuracy on train set: 0.99 ± 0.01
Probe accuracy on hidden test set: 1.00


## Visualization

Finally, we visualize the probe's predictions on a sample of the unseen test set natively in the notebook. This compares the actual board position directly against the prediction class from our Logistic Regression logic. Red text indicates an error, while green indicates correct classification.

In [28]:
from IPython.display import display, HTML
import chess.svg

print("Visualizing probe predictions on TEST SET...")

try:
    # Get the probability that the position is in check (class 1) from the test set
    probs = probe.predict_proba(X_test)[:, 1]

    # Find some indices to plot (within the test set arrays)
    test_in_check_idx = np.where(y_test == 1)[0][:5]
    test_not_in_check_idx = np.where(y_test == 0)[0][:5]

    html_out = "<div style='display: flex; flex-direction: column; gap: 20px;'>"

    def render_group(test_indices, title):
        html = f"<h3>{title}</h3><div style='display: flex; flex-wrap: wrap; gap: 15px;'>"
        for t_idx in test_indices:
            # Map back to the original index in `boards`
            orig_idx = idx_test[t_idx] 
            
            board = boards[orig_idx].pc_board
            prob = probs[t_idx]
            
            # Color the probability green if correct, red if incorrect
            is_correct = (prob > 0.5 and y_test[t_idx] == 1) or (prob <= 0.5 and y_test[t_idx] == 0)
            color = "green" if is_correct else "red"
            
            svg = chess.svg.board(board, size=200)
            html += f"""
            <div style="border: 1px solid #ccc; padding: 10px; border-radius: 8px; text-align: center; background: white; color: black;">
                {svg}
                <div style="margin-top: 10px; font-family: sans-serif;">
                    <b>Prob (is_check):</b> <span style="color: {color}">{prob:.1%}</span><br/>
                    <b>True Label:</b> {bool(y_test[t_idx])}
                </div>
            </div>
            """
        html += "</div>"
        return html

    html_out += render_group(test_in_check_idx, "5 Test Positions IN CHECK")
    html_out += render_group(test_not_in_check_idx, "5 Test Positions NOT IN CHECK")
    html_out += "</div>"

    display(HTML(html_out))

except NameError as e:
    print(f"Error: Make sure you ran the previous training cell first. ({e})")

Visualizing probe predictions on TEST SET...


## Visualizing Attention Outputs

Here we extract the inner attention matrices from Leela's ONNX-converted Transformer layer using `nnsight`. Since King safety is directly correlated to the concept of check, we map how much the `King` query token attends to other pieces (keys) on the board!

In [52]:
from IPython.display import display, HTML

# Define available layers and heads in the model
AVAILABLE_ATTENTION = {
    "layers": [f"encoder{i}" for i in range(10)], # encoder0 up to encoder9
    "heads": list(range(24)),
    "description": "24 heads per layer across 10 encoder layers."
}

def plot_attention_for_square(board_idx, query_square, layer_name="encoder7", head_idx=None):
    """
    Extracts and visualizes the attention weights for a given square on the board.
    If head_idx is None, averages across all 24 heads.
    """
    board_demo = boards[board_idx]
    
    # Forward pass on ONE board
    single_input = inputs[board_idx:board_idx+1]
    
    with nnsight_model.trace(single_input) as tracer:
        # Construct module path dynamically
        module_path = f"{layer_name}/mha/QK/softmax"
        layer = getattr(nnsight_model._lc0_model, module_path)
        attn_out = layer.output.save()
        
    weights = attn_out if isinstance(attn_out, torch.Tensor) else attn_out.value 
    weights = weights[0].cpu().numpy() # [24, 64, 64]
    
    if head_idx is None:
        mean_attn = weights.mean(axis=0)[query_square] # average across all 24 heads
    else:
        mean_attn = weights[head_idx][query_square] # specific head

    # Normalize between 0 and 1 so it fits a color scale smoothly
    min_val, max_val = mean_attn.min(), mean_attn.max()
    norm_attn = (mean_attn - min_val) / (max_val - min_val + 1e-9)

    colors = {}
    for sq in range(64):
        val = float(norm_attn[sq])
        # White to Red heatmap background
        color_hex = f"#{int(255):02x}{int(255*(1-val)):02x}{int(255*(1-val)):02x}"
        colors[sq] = color_hex

    svg = chess.svg.board(
        board_demo.pc_board,
        fill=colors,
        size=400
    )

    # Put a nice title above
    head_title = "Average across all 24 heads" if head_idx is None else f"Head {head_idx}"
    html_out = f"<h3>Attention Map: {layer_name}, {head_title} | Query square: {chess.square_name(query_square).upper()}</h3>"
    html_out += svg
    display(HTML(html_out))

print(f"Available layers to choose from: {AVAILABLE_ATTENTION['layers']}")
print(f"Available heads to choose from: {AVAILABLE_ATTENTION['heads'][0]} to {AVAILABLE_ATTENTION['heads'][-1]}")

Available layers to choose from: ['encoder0', 'encoder1', 'encoder2', 'encoder3', 'encoder4', 'encoder5', 'encoder6', 'encoder7', 'encoder8', 'encoder9']
Available heads to choose from: 0 to 23


In [62]:
demo_idx = next(
    (i for i in in_check_idx if boards[i].pc_board.turn == chess.WHITE),
    in_check_idx[0] # fallback
)

print("Visualizing White King query:")
w_king_sq = boards[demo_idx].pc_board.king(chess.WHITE)
plot_attention_for_square(demo_idx, w_king_sq, layer_name="encoder7", head_idx=None)

print("\nVisualizing Black King query (on the same board):")
b_king_sq = boards[demo_idx].pc_board.king(chess.BLACK)
plot_attention_for_square(demo_idx, b_king_sq, layer_name="encoder7", head_idx=None)

Visualizing White King query:



Visualizing Black King query (on the same board):
